# YOLOv8s training -- pool 60 synthetic dataset

Reproduces `05_train.py --pool 60 --model yolov8s.pt` from the local
pipeline exactly: same hyperparameters, same F1-operating-point extraction
logic (`pipeline_common.py`, inlined below), same output layout -- so the
results are directly comparable to the local yolo11s run.

### Before running
1. Attach the e-waste dataset under **Input** (already done if you see it listed).
2. **Settings -> Accelerator: GPU** (T4 x2 or P100).
3. **Settings -> Internet: On** (needed for `pip install` and the pretrained checkpoint).

### To run overnight unattended
Use **Save Version -> Save & Run All (Commit)**, NOT the interactive Run All.
The commit runs as a background batch job on Kaggle's servers, so it survives
closing the browser.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, json, time, shutil, zipfile
from pathlib import Path
import numpy as np
from ultralytics import YOLO

INPUT_DIR = Path('/kaggle/input')
WORK = Path('/kaggle/working')

# ---- locate the dataset ----
# Recursive: Kaggle nests uploads at an unpredictable depth
# (e.g. /kaggle/input/datasets/<user>/<slug>/dataset_pool60/data.yaml),
# so do not assume a fixed number of directory levels.
found = sorted(INPUT_DIR.rglob('data.yaml'))
assert found, f'no data.yaml found anywhere under {INPUT_DIR} -- is the dataset attached?'
SRC_ROOT = found[0].parent
print('found dataset at:', SRC_ROOT)

# ---- copy it into the writable working directory ----
# /kaggle/input is mounted READ-ONLY. Ultralytics needs to write label .cache
# files next to the labels, and we need to rewrite data.yaml's `path:` field,
# so neither can happen in place. ~220 MB against a 20 GB working quota.
DATA_ROOT = WORK / 'dataset_pool60'
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
print('copying dataset into writable working dir ...')
shutil.copytree(SRC_ROOT, DATA_ROOT)
print('copied to:', DATA_ROOT)

# ---- rewrite data.yaml ----
# The local build baked an absolute Windows path into `path:`. Rewrite the
# whole file rather than patching it, so it cannot carry stale fields.
data_yaml = DATA_ROOT / 'data.yaml'
data_yaml.write_text(
    f'path: {DATA_ROOT.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    '\n'
    'nc: 1\n'
    "names: ['ewaste_contaminant']\n",
    encoding='utf-8')
print()
print(data_yaml.read_text(encoding='utf-8'))

# ---- sanity check before committing to a long run ----
n_train = len(list((DATA_ROOT / 'images' / 'train').glob('*.jpg')))
n_val = len(list((DATA_ROOT / 'images' / 'val').glob('*.jpg')))
print(f'train images: {n_train}   val images: {n_val}')
assert n_train > 0 and n_val > 0, 'dataset copied but images are missing'

In [ ]:
# ---- pipeline_common.py, inlined so this notebook is self-contained ----
# Identical to the local module: the F1-optimal synthetic confidence is what
# the paper carries into the real-image evaluation, so if two models computed
# it differently the comparison between them would be meaningless.

def f1_curve(metrics):
    box = getattr(metrics, 'box', None)
    x = getattr(box, 'px', None)
    y = getattr(box, 'f1_curve', None)
    if x is not None and y is not None:
        y = np.asarray(y, dtype=float)
        if y.ndim > 1:
            y = y.mean(axis=0)
        return np.asarray(x, dtype=float), y
    for entry in getattr(metrics, 'curves_results', []) or []:
        try:
            cx, cy, xlabel, ylabel = entry
        except (TypeError, ValueError):
            continue
        if 'f1' not in str(ylabel).lower() or 'confidence' not in str(xlabel).lower():
            continue
        cy = np.asarray(cy, dtype=float)
        if cy.ndim > 1:
            cy = cy.mean(axis=0)
        return np.asarray(cx, dtype=float), cy
    return None, None


def best_f1_point(metrics):
    x, y = f1_curve(metrics)
    if x is None or y is None or len(x) != len(y):
        return None, None
    i = int(np.argmax(y))
    return float(x[i]), float(y[i])


def write_f1_curve(out_dir, metrics):
    cx, cy = f1_curve(metrics)
    if cx is None or cy is None or len(cx) != len(cy):
        return False
    (out_dir / 'f1_curve.csv').write_text(
        'confidence,f1\n' + '\n'.join(f'{a:.4f},{b:.6f}' for a, b in zip(cx, cy)),
        encoding='utf-8')
    return True

In [ ]:
# ---- identical config to: 05_train.py --pool 60 --model yolov8s.pt ----

POOL = 60
MODEL = 'yolov8s.pt'
EPOCHS = 120
IMG_SIZE = 640
BATCH = 16
SEED = 0
RUN_NAME = f'pool{POOL}'
PROJECT = '/kaggle/working/runs/detect'

model = YOLO(MODEL)

t0 = time.time()
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    patience=30,
    seed=SEED,
    deterministic=True,
    device=0,
    # --- augmentation tuned for small, partly buried objects ---
    mosaic=1.0,
    close_mosaic=15,
    scale=0.5,
    translate=0.2,
    fliplr=0.5,
    flipud=0.3,
    degrees=15.0,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    erasing=0.3,
    copy_paste=0.0,        # the data is already built by compositing
    project=PROJECT,
)
train_seconds = time.time() - t0
print(f'\ntraining took {train_seconds/60:.1f} min')

In [ ]:
print('--- validation on the SYNTHETIC split ---')
metrics = model.val()
conf, f1 = best_f1_point(metrics)

out_dir = Path(PROJECT) / RUN_NAME
write_f1_curve(out_dir, metrics)

summary = {
    'pool': POOL,
    'seed': SEED,
    'model': MODEL,
    'epochs': EPOCHS,
    'dataset': 'dataset_pool60',
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'best_f1': f1,
    'best_f1_conf': conf,
    'train_seconds': round(train_seconds, 1),
    'trained_on': 'kaggle',
}
(out_dir / 'synthetic_summary.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8')

print('\nSynthetic validation summary')
for k, v in summary.items():
    print(f'  {k:<14} {v}')

In [ ]:
# ---- package the run for download ----
# Only what 06_evaluate.py and the paper need, so the download stays small.

download_dir = WORK / 'pool60_for_download'
if download_dir.exists():
    shutil.rmtree(download_dir)
(download_dir / 'weights').mkdir(parents=True, exist_ok=True)

shutil.copy(out_dir / 'weights' / 'best.pt', download_dir / 'weights' / 'best.pt')
shutil.copy(out_dir / 'synthetic_summary.json', download_dir / 'synthetic_summary.json')
for optional in ('f1_curve.csv', 'results.csv', 'args.yaml'):
    if (out_dir / optional).exists():
        shutil.copy(out_dir / optional, download_dir / optional)

zip_out = WORK / 'pool60_trained.zip'
if zip_out.exists():
    zip_out.unlink()
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in download_dir.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(download_dir))

print('packaged:', zip_out, f'({zip_out.stat().st_size/1e6:.1f} MB)')
print('\nDownload this from the Output panel once the run completes.')

# Free the working directory of the dataset copy so the output stays small.
shutil.rmtree(DATA_ROOT, ignore_errors=True)
print('cleaned up the dataset copy from /kaggle/working')

### After downloading `pool60_trained.zip`

Extract it locally into `runs/detect/pool60/` so the layout is:

```
runs/detect/pool60/
  weights/best.pt
  synthetic_summary.json
  f1_curve.csv
  results.csv
```

Then score it against the same real-photo test sets everything else uses:

```bash
python 06_evaluate.py --pool 60
```